# Atlas World — Kaggle precompute → GitHub → Cloudflare

This notebook keeps the heavy GeoBoundaries download, geometry normalization, antimeridian split, Web Mercator projection, MVT encoding, audit, and archive generation in Kaggle. The browser only serves the validated static files from the repository.

## One-time Kaggle setup

Enable **Internet** in the notebook settings. Add one Kaggle Secret named `GITHUB_TOKEN`. The value must be a fine-grained GitHub token scoped only to `crestog/unemployed-nigg-` with repository **Contents: Read and write**. Never paste the token into a cell or chat; the next cell reads it through Kaggle Secrets.

The notebook publishes only after the builder and audit pass. A rejected polar detail feature is recorded in the release audit; world-spanning, invalid-after-repair, or excessive-replication failures stop the run before GitHub is changed.

In [ ]:
import os
from datetime import datetime, timezone
from pathlib import Path
from kaggle_secrets import UserSecretsClient

try:
    token = UserSecretsClient().get_secret('GITHUB_TOKEN')
except Exception as exc:
    raise RuntimeError('Add Kaggle Secret GITHUB_TOKEN before continuing. Do not paste it into the notebook source.') from exc
if not token:
    raise RuntimeError('Kaggle Secret GITHUB_TOKEN is empty.')
os.environ['GITHUB_TOKEN'] = token
REPO_DIR = Path('/kaggle/working/atlas-repo')
REPO_URL = 'https://github.com/crestog/unemployed-nigg-.git'
RELEASE_ID = f"world-global-geoboundaries-kaggle-{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}"
ARCHIVE = Path('/kaggle/working') / f'atlas-world-{RELEASE_ID}.tar.gz'
print({'repo': str(REPO_DIR), 'releaseId': RELEASE_ID, 'archive': str(ARCHIVE), 'tokenLoaded': True})

## 1. Clone the latest repository

This resets the Kaggle checkout to the current `main` branch, so the notebook always uses the newest builder and frontend code.

In [ ]:
!rm -rf /kaggle/working/atlas-repo
!git clone --depth 1 --branch main https://github.com/crestog/unemployed-nigg- /kaggle/working/atlas-repo
!git -C /kaggle/working/atlas-repo log -1 --oneline

## 2. Install build-only dependencies

These packages are used only inside Kaggle. They are not required by the Cloudflare Worker or the mobile browser.

In [ ]:
import os, subprocess, sys
from pathlib import Path
VENV_DIR = Path('/kaggle/working/atlas-venv')
if not VENV_DIR.exists():
    subprocess.run([sys.executable, '-m', 'venv', str(VENV_DIR)], check=True)
VENV_PYTHON = VENV_DIR / 'bin' / 'python'
subprocess.run([str(VENV_PYTHON), '-m', 'pip', 'install', '--quiet', '--upgrade', 'ijson', 'shapely', 'antimeridian', 'mapbox-vector-tile', 'pyclipper', 'protobuf', 'pyproj', 'numpy'], check=True)
print('Isolated build environment ready:', VENV_PYTHON)

## 3. Build the immutable release and run the source audit

The wrapper downloads the official CGAZ ADM1/ADM2 snapshots, prints source SHA-256 hashes, runs the hardened builder with bounded parallel layer processing, emits polygon and point-label MVT layers, and writes JSON/CSV geometry audits. It then fails on disallowed world-spanning or invalid geometry violations.

In [ ]:
import os, subprocess
env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'
cpu_count = os.cpu_count() or 1
workers = max(1, min(4, cpu_count))
gpu_probe = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True, check=False)
if gpu_probe.returncode == 0 and gpu_probe.stdout.strip():
    print('GPU detected, but exact topology/MVT path remains CPU-safe; using native pyproj projection plus', workers, 'tile workers.')
    print(gpu_probe.stdout.strip())
else:
    print('No usable GPU detected; using', workers, 'CPU tile workers.')
subprocess.run([
    str(VENV_PYTHON), '/kaggle/working/atlas-repo/scripts/kaggle_build_atlas_world.py',
    '--repo-dir', '/kaggle/working/atlas-repo',
    '--repo-url', 'https://github.com/crestog/unemployed-nigg-.git',
    '--branch', 'main',
    '--release-id', os.environ['RELEASE_ID'],
    '--archive', f"/kaggle/working/atlas-world-{os.environ['RELEASE_ID']}.tar.gz",
    '--skip-install',
    '--workers', str(workers),
    '--parallel-layers',
], env=env, check=True)
print('Build process completed.')

## 4. Inspect the generated audit and archive

This cell is informational. It should show four non-empty runtime layers: `adm1`, `adm1Labels`, `adm2`, and `adm2Labels`. The `polarRejectedCount` values are expected policy output for detail geometries outside the safe vector latitude; they are not silently clamped into invalid Web Mercator geometry.

In [ ]:
import json
release_dir = REPO_DIR / 'client' / 'public' / 'data' / 'world-mvt' / RELEASE_ID
manifest = json.loads((release_dir / 'manifest.json').read_text())
audit_summary = json.loads((release_dir / 'geometry-audit-summary.json').read_text())
print(json.dumps({'releaseId': manifest['releaseId'], 'geometryPolicy': manifest['geometryPolicy'], 'audit': audit_summary, 'archiveBytes': ARCHIVE.stat().st_size}, indent=2))
assert set(['adm1', 'adm1Labels', 'adm2', 'adm2Labels']).issubset(manifest['layers'])
assert ARCHIVE.exists() and ARCHIVE.stat().st_size > 0

## 5. Confirm the pushed commit

The wrapper commits and pushes only after validation. This explicit check confirms the commit reached the remote without printing the token or embedding it in the remote URL.

In [ ]:
!git -C /kaggle/working/atlas-repo status --short
!git -C /kaggle/working/atlas-repo log -1 --oneline
print('The validated release was pushed by the wrapper. The mutable manifest now points at:', RELEASE_ID)

## 6. Monitor GitHub Actions, then test the live site

The existing GitHub Actions workflow deploys `main` to Cloudflare. This cell polls the public Actions API briefly, prints the workflow URL, and prints the live Atlas URL. If Actions is still queued, wait and refresh the live URL after it completes.

In [ ]:
import time
import urllib.request

actions_url = 'https://github.com/crestog/unemployed-nigg-/actions'
api_url = 'https://api.github.com/repos/crestog/unemployed-nigg-/actions/runs?branch=main&per_page=5'
latest = None
for attempt in range(12):
    try:
        request = urllib.request.Request(api_url, headers={'Accept': 'application/vnd.github+json', 'User-Agent': 'atlas-kaggle-builder'})
        with urllib.request.urlopen(request, timeout=20) as response:
            runs = json.loads(response.read().decode()).get('workflow_runs', [])
        latest = next((run for run in runs if RELEASE_ID in (run.get('head_commit', {}) or {}).get('message', '')), runs[0] if runs else None)
        if latest:
            print({'status': latest.get('status'), 'conclusion': latest.get('conclusion'), 'html_url': latest.get('html_url')})
            if latest.get('status') == 'completed':
                break
    except Exception as exc:
        print('Actions status not available yet:', type(exc).__name__)
    time.sleep(10)

print('Actions:', actions_url)
print('Live test:', 'https://unemployed-nigg.sahudevansh482.workers.dev/#world=1')
print('Expected current release:', RELEASE_ID)

## 7. Manual mobile acceptance test

After Actions reports success, open the live test URL on the phone and test one continuous globe composition: pan/pinch/rotate at country scale, the Middle East and dateline, India/Japan and China/Russia, then approach the safe polar threshold. Confirm that country/ADM layers do not stack incorrectly, labels are not duplicated, and no dark rectangular or world-spanning fill appears. Polar administrative detail remains deliberately limited by the documented Web Mercator policy rather than claimed as fully solved.